# TD-DFTB-based NBRA-workflow for DFTB+

In this tutorial, we demonstrate some some higher-level functions to streamline DFTB+ calculations of the properties needed for NA-MD simulations. We will use these functions to define a workflow for NBRA calculations with DFTB+ at the TD-DFTB level of electronic structure. While the resulting function is a prototype for the non-NBRA calculations, at this point we will only focus on its use for NBRA calculations.

## Table of contents
<a name="toc"></a>
1. [Writing Gen-files](#1)
2. [Writing input file for DFTB+, `dftb_in.hsd`](#2)
3. [Needed Slater-Koster files and system-specific parameters](#3)
4. [Run DFTB+ calculations](#4)

   4.1. [To generate H and S](#4.1)
   
   4.2. [To do SCF and TD-DFT calculations](#4.2)
   
   4.3. [The overlap for the doubled-molecule - ODIN](#4.3)
   
   4.4. [Check time-overlap](#4.4)
   
   4.5. [Extracting MO/CI data in the required format](#4.5)
   
5. [Putting everything together](#5)
    
   5.1. [Develop the function](#5.1)
   
   5.2. [Test the funciton](#5.2)
   
   5.3. [Test Libra implementation](#5.3)
   

### A. Learning objectives

* To create gen files for DFTB+ calculations
* To create DFTB+ input files 
* To run ODIN calculations of atomic overlaps
* To compute time-overlaps of AOs using ODIN
* To extract the key infromation from the TD-DFTB calculations from DFTB+ output (energies, MOs, etc.)
* To compute NACs with TD-DFTB excited states
* To execute NBRA workflow with DFTB+


### B. Use cases

* process the DFTB+ calculations results
* compute wavefunction time-overlaps with DFTB+
* NBRA workflow with DFTB+


### C. Functions

- `libra_py`
  - `packages`
    - `dftbplus`
      - `methods`
        - [`create_odin_inp`](#create_odin_inp-1)
        - [`dftb_compute_adi`](#dftb_compute_adi-1)
        - [`make_dftb_input`](#make_dftb_input-1)
        - [`read_dftb_orbital_info`](#read_dftb_orbital_info-1)
        - [`read_overlap_matrix`](#read_overlap_matrix-1)
        - [`read_spx_mappings`](#read_spx_mappings-1)
        - [`run_dftb`](#run_dftb-1)
        - [`run_odin`](#run_odin-1)
        - [`write_dftb_gen`](#write_dftb_gen-1)


In [1]:
import os
import sys
import re
import copy
import numpy as np
from liblibra_core import MATRIX, CMATRIX, Py2Cpp_int, Cpp2Py

# Fisrt, we add the location of the library to test to the PYTHON path
from libra_py.packages.dftbplus import methods as DFTB_methods
from libra_py.packages.cp2k import methods as cp2k
import libra_py.citools.ci as ci
from libra_py import data_conv
from libra_py import units

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

## 1. Writing Gen-files
<a name="1"></a>[Back to TOC](#toc)

<a name="write_dftb_gen-1"></a>

In [2]:
help(DFTB_methods.write_dftb_gen)

Help on function write_dftb_gen in module libra_py.packages.dftbplus.methods:

write_dftb_gen(filename, atom_labels, coordinates, periodic=False)
    Write a DFTB+ .gen file.
    
    Parameters
    ----------
    filename : str
        Output file name.
    atom_labels : list of str
        Atomic symbols, length N.
    coordinates : array-like, shape (3N, 1)
        Flattened Cartesian coordinates:
        [x1,y1,z1,x2,y2,z2,...]^T  (Angstrom)
    periodic : bool
        True -> periodic system (S)
        False -> cluster (C)
    
    Example
    -------
    > atom_labels = ["O", "H", "H"]
    > 
    > coordinates = np.array([
    > 0.000000, 0.000000, 0.000000,
    > 0.758602, 0.000000, 0.504284,
    > -0.758602, 0.000000, 0.504284
    > ]).reshape(-1, 1)
    >
    > write_dftb_gen("water.gen", atom_labels, coordinates)



In [3]:
atom_labels = ["O", "H", "H"]

coordinates = np.array([
    0.000000, 0.000000, 0.000000,
    0.758602, 0.000000, 0.504284,
   -0.758602, 0.000000, 0.504284
]).reshape(-1, 1)

DFTB_methods.write_dftb_gen("water.gen", atom_labels, coordinates)

DFTB+ GEN file written to water.gen


## 2. Writing input file for DFTB+, `dftb_in.hsd`
<a name="2"></a>[Back to TOC](#toc)

<a name="make_dftb_input-1"></a>

In [4]:
help(DFTB_methods.make_dftb_input)

Help on function make_dftb_input in module libra_py.packages.dftbplus.methods:

make_dftb_input(params)
    Generate a DFTB+ input file ("dftb_in.hsd") for ground-state SCC-DFTB and
    excited-state TD-DFTB (Casida formalism) calculations.
    
    This function constructs a complete DFTB+ input file using parameters
    provided in the ``params`` dictionary. Missing parameters are automatically
    filled with reasonable defaults. The generated file is written to the
    current working directory as "dftb_in.hsd".
    
    The input includes the following sections:
    
        - Geometry (GenFormat)
        - Driver
        - Hamiltonian (SCC-DFTB)
        - ExcitedState (Casida TD-DFTB)
        - Options
        - Analysis
    
    Parameters
    ----------
    params : dict
        Dictionary containing DFTB+ input parameters. All keys are optional.
        The following keys are recognized:
    
        Geometry and general:
        ---------------------
        gen_file : str, o

In [5]:
dftb_params = {
    "gen_file" : "x1.gen",
    "sk_prefix" : "../mio/FinalSK/",
    "Driver" : "{}",
    "MaxAngularMomentum" : """{ O = "p" 
                          H = "s" 
                     }
                     """,
    "Symmetry" : "Singlet",
    "NrOfExcitations" : 5,
    "StateOfInterest" : 1,
    "WriteSPTransitions" : "Yes",
    "WriteXplusY" : "Yes",
    "WriteXplusYAscii" : "Yes",
    "StateCouplings" : "{0 2}",
    
    "WriteAutotestTag" : "Yes",
    "WriteHS" : "Yes",
    "WriteEigenvectors" : "Yes",
    "EigenvectorsAsText" : "Yes",
    "PrintForces" : "Yes"
}
DFTB_methods.make_dftb_input(dftb_params)

## 3. Needed Slater-Koster files and system-specific parameters
<a name="3"></a>[Back to TOC](#toc)

We need to use specially-generated SK files. Here, we have two sets provided by Thomas Niehaus:

- mio-0-1_xSK.tgz
- ob2-1-1_xSK.tgz

In [6]:
#!tar -xf mio-0-1_xSK.tgz

Let's also define the path to the DFTB+ code. Change it, depending on the file system where you are running this tutorial

In [7]:
DFTB_EXE = "/home/alexvakimov/opt/dftb+/bin/dftb+"
ODIN_EXE = "/home/alexvakimov/SOFTWARE/odin/odin"

## 4. Run DFTB+ calculations 
<a name="4"></a>[Back to TOC](#toc)

### 4.1. To generate H and S
<a name="4.1"></a>[Back to TOC](#toc)

<a name="run_dftb-1"></a>

In [8]:
help(DFTB_methods.run_dftb)

Help on function run_dftb in module libra_py.packages.dftbplus.methods:

run_dftb(coords, params)
    Run a DFTB+ calculation for a given molecular geometry.
    
    This function prepares and executes a DFTB+ calculation in a dedicated
    working directory. It performs the following steps:
    
        1. Creates (if necessary) a working directory.
        2. Writes a GEN-format geometry file using the provided coordinates.
        3. Generates the DFTB+ input file ("dftb_in.hsd").
        4. Executes the DFTB+ binary.
        5. Returns to the original working directory.
    
    The function is designed for automated workflows such as molecular
    dynamics, nonadiabatic dynamics, or high-throughput simulations.
    
    Parameters
    ----------
    coords : ndarray
        Cartesian atomic coordinates. Expected shape is (3*N,) or (3*N, 1),
        where N is the number of atoms. Coordinates are assumed to be in
        atomic units (Bohr) and are internally converted to Angstrom

In [9]:
atom_labels = ["O", "H", "H"]

coordinates = np.array([
    0.000000, 0.000000, 0.000000,
    0.758602, 0.000000, 0.504284,
   -0.758602, 0.000000, 0.504284
]).reshape(-1, 1) * units.Angst 

prms1 = { "labels" : atom_labels,
          "exe": DFTB_EXE,
          "dftb_run_params" : dftb_params,
          "working_directory" : "calc1",
        }
DFTB_methods.run_dftb(coordinates, prms1)

DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

--------------------------------------------------------------------------------
WARN

ERROR STOP 

Error termination. Backtrace:
#0  0x5cde36b0488e in __dftbp_common_globalenv_MOD_abortprogram
	at /home/alexvakimov/SOFTWARE/dftbplus/src/dftbp/common/globalenv.F90:195
#1  0x5cde36ca966a in __dftbp_dftbplus_mainio_MOD_writehsandstop
	at /home/alexvakimov/SOFTWARE/dftbplus/src/dftbp/dftbplus/mainio.F90:4334
#2  0x5cde36c8e4e8 in __dftbp_dftbplus_main_MOD_processgeometry
	at /home/alexvakimov/SOFTWARE/dftbplus/src/dftbp/dftbplus/main.F90:1477
#3  0x5cde36c93c8a in __dftbp_dftbplus_main_MOD_rundftbplus
	at /home/alexvakimov/SOFTWARE/dftbplus/src/dftbp/dftbplus/main.F90:252
#4  0x5cde36b03ad1 in dftbplus
	at /home/alexvakimov/SOFTWARE/dftbplus/app/dftb+/dftbplus.F90:33
#5  0x5cde36affdee in main
	at /home/alexvakimov/SOFTWARE/dftbplus/app/dftb+/dftbplus.F90:11


### 4.2. To do SCF and TD-DFT calculations
<a name="4.2"></a>[Back to TOC](#toc)

In [10]:
dftb_params.update({"WriteHS" : "No"})
prms2 = { "labels" : atom_labels,
          "exe": DFTB_EXE,
          "dftb_run_params" : dftb_params,
          "working_directory" : "calc2",
        }
DFTB_methods.run_dftb(coordinates, prms2)

DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

--------------------------------------------------------------------------------
WARN

### 4.3. The overlap for the doubled-molecule - ODIN
<a name="4.3"></a>[Back to TOC](#toc)

In [11]:
double_labels = atom_labels*2
double_coords = np.concatenate( (coordinates, coordinates), axis=0)

print(double_labels)
print(double_coords)

['O', 'H', 'H', 'O', 'H', 'H']
[[ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 1.43354991]
 [ 0.        ]
 [ 0.95295858]
 [-1.43354991]
 [ 0.        ]
 [ 0.95295858]
 [ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 1.43354991]
 [ 0.        ]
 [ 0.95295858]
 [-1.43354991]
 [ 0.        ]
 [ 0.95295858]]


Next, we will use two auxiliary functions: `create_odin_inp` 
<a name="create_odin_inp-1"></a>

In [12]:
help(DFTB_methods.create_odin_inp)

Help on function create_odin_inp in module libra_py.packages.dftbplus.methods:

create_odin_inp(params)
    Generate the input string for the ODIN overlap program.
    
    This function constructs the text input required by the ODIN code
    (https://github.com/thomas-niehaus/odin) for computing overlap and
    Hamiltonian-related quantities from Slater–Koster files and a GEN
    geometry file.
    
    The function:
    
        1. Reads the provided GEN file.
        2. Extracts the list of chemical elements from the second line.
        3. Appends the corresponding maximum angular momentum (ℓ_max)
           values for each element.
        4. Returns the complete ODIN input as a formatted string.
    
    Parameters
    ----------
    params : dict
        Dictionary containing ODIN input parameters.
    
        Optional keys
        -------------
        filename : str, optional
            Path to the GEN-format geometry file for which overlaps
            will be computed.
   

and `run_odin`
<a name="run_odin-1"></a>

In [13]:
help(DFTB_methods.run_odin)

Help on function run_odin in module libra_py.packages.dftbplus.methods:

run_odin(params)
    Execute the ODIN program to compute atomic orbital overlap matrices.
    
    This function prepares the required ODIN input file ("odin.inp"),
    runs the ODIN executable, and computes overlap-related quantities
    based on the provided GEN geometry file and Slater–Koster parameters.
    
    Specifically, the function:
    
        1. Generates the ODIN input string using ``create_odin_inp``.
        2. Writes the input to the file "odin.inp".
        3. Executes the ODIN program using standard input redirection.
        4. Produces overlap output files (e.g., "oversqr.dat").
    
    ODIN is a post-processing tool for DFTB that computes atomic orbital
    overlap matrices and related quantities from Slater–Koster files and
    molecular geometries.
    
    Parameters
    ----------
    params : dict
        Dictionary containing ODIN execution parameters.
    
        Optional keys
     

In [14]:
wd = "calc3"

# Create working directory, if doesn't exist
if not os.path.exists(wd):
    os.mkdir(wd)
# Go into that directory
os.chdir(wd)

# Make double geometry gen file
DFTB_methods.write_dftb_gen("doubled_geometry.gen", double_labels, double_coords/units.Angst)

# Make input for Odin:
odin_params = {
    "ODIN_EXE": ODIN_EXE,
    "filename":"doubled_geometry.gen",
    "slakos_prefix" : "../mio/FinalSK/",
    "max_ang_mom": {"H":1, "O":2},
}
DFTB_methods.create_odin_inp(odin_params)

# Run Odin
DFTB_methods.run_odin(odin_params)

# Go back to the original directory
os.chdir("../")

DFTB+ GEN file written to doubled_geometry.gen
['O', 'H']
['O', 'H']
 ** odin (t.niehaus, based on dylcao by seifert,porezag,blaudeck) **
 Creates dftb overlap matrix and stores the result in oversqr.dat
 ** Version 01  02.05.2025 **
 enter filename for input structure
 infile :doubled_geometry.gen
 enter prefix for Slater-Koster path
 Prefix is ../mio/FinalSK/
 enter separator of Slater-Koster file names
 Separator is -
 enter suffix of Slater-Koster file
 Suffix is .skf
 enter            2  * lmax
 skfile for pair            1           1  :../mio/FinalSK/O-O.skf
 skfile for pair            1           2  :../mio/FinalSK/O-H.skf
 skfile for pair            2           1  :../mio/FinalSK/H-O.skf
 skfile for pair            2           2  :../mio/FinalSK/H-H.skf
 Results in oversqr.dat
 Enjoy!


### 4.4. Check time-overlap
<a name="4.4"></a>[Back to TOC](#toc)

First, let's use one of the auxiliary functions to get the dimension of the overlap matrix (the number of AOs):
<a name="read_spx_mappings-1"></a>

In [15]:
rpa_map, _, _ = DFTB_methods.read_spx_mappings("calc2/SPX.DAT")
ndim = rpa_map.shape[0]
print(ndim)

6


Then, we can read the matrix. Note that since this is the overlap matrix for the doubled molecule, we use doubled dimension `2*ndim`.

Once it is read, we read sub-blocks of the bigger overlap matrix. The diagonal blocks are the self-overlaps (of the same geometry with itself) just for different geometries (first and second). 

The off-diagonal blocks are the overlaps of AOs between two geometries first and second.

Since in our input above the firs and second geometries are identical, we expect to have all the block matrices to be alike.
<a name="read_overlap_matrix-1"></a>

In [16]:
## Overlap matrix
S = DFTB_methods.read_overlap_matrix('calc3/oversqr.dat', 2*ndim)
print(S.shape)
print(S)
print("S00")
print(S[:ndim, :ndim])
print("S01")
print(S[:ndim, ndim:2*ndim])
print("S10")
print(S[ndim:2*ndim, :ndim])
print("S11")
print(S[ndim:2*ndim, ndim:2*ndim])

(12, 12)
[[ 1.          0.          0.          0.          0.46501291  0.46501291
   1.          0.          0.          0.          0.46501291  0.46501291]
 [ 0.          1.          0.          0.          0.          0.
   0.          1.          0.          0.          0.          0.        ]
 [ 0.          0.          1.          0.          0.22860337  0.22860337
   0.          0.          1.          0.          0.22860337  0.22860337]
 [ 0.          0.          0.          1.          0.34389149 -0.34389149
   0.          0.          0.          1.          0.34389149 -0.34389149]
 [ 0.46501291  0.          0.22860337  0.34389149  1.          0.19317855
   0.46501291  0.          0.22860337  0.34389149  1.          0.19317855]
 [ 0.46501291  0.          0.22860337 -0.34389149  0.19317855  1.
   0.46501291  0.          0.22860337 -0.34389149  0.19317855  1.        ]
 [ 1.          0.          0.          0.          0.46501291  0.46501291
   1.          0.          0.          

### 4.5. Extracting MO/CI data in the required format
<a name="4.5"></a>[Back to TOC](#toc)

We use the `DFTB_methods.read_dftb_orbital_info` function to read the results of the TD-DFTB calculations in the format expected in the workflow:
<a name="read_dftb_orbital_info-1"></a>

In [17]:
help(DFTB_methods.read_dftb_orbital_info)

Help on function read_dftb_orbital_info in module libra_py.packages.dftbplus.methods:

read_dftb_orbital_info(params_)
    Read molecular orbital (MO) information, configurations, and CI amplitudes
    from DFTB+ excited-state output files.
    
    This function extracts the active-space MO coefficients and excited-state
    configuration interaction (CI) information from DFTB+ linear-response
    (Casida/RPA) output files.
    
    Parameters
    ----------
    params_ : dict
        Dictionary of input parameters. Recognized keys:
    
        source_directory : str, optional, default="calc"
            Directory containing the DFTB+ output files:
                - SPX.DAT      : orbital excitation mappings
                - eigenvec.bin : MO coefficient matrix
                - XplusY.DAT   : excitation energies and CI vectors
    
        orbital_space : list of int, optional, default=None
            List of molecular orbital indices (1-based indexing) defining
            the ac

In [18]:
info, mo1, data1 = DFTB_methods.read_dftb_orbital_info({"nstates":5, "orbital_space":None, "source_directory":"calc2" })

In [19]:
print(mo1)
print(data1[0])
print(data1[1])
print(data1[2])
print(info)

[[ 8.33685274e-01 -2.77555756e-17  2.69865325e-01 -1.23098780e-16
  -4.35270599e-16 -9.44042104e-01]
 [ 0.00000000e+00 -1.63757896e-15 -4.16333634e-16 -1.00000000e+00
   1.11022302e-16  0.00000000e+00]
 [-1.14774356e-03 -6.03092773e-17 -8.94142966e-01  3.58878678e-16
  -3.08318105e-16 -5.99824914e-01]
 [ 6.95067645e-20 -6.43528029e-01  3.40551072e-17  8.13345491e-16
  -1.00030346e+00  4.59762952e-16]
 [ 1.58238665e-01 -3.87758479e-01 -1.74442914e-01  7.48675878e-16
   8.52277813e-01  8.40541619e-01]
 [ 1.58238665e-01  3.87758479e-01 -1.74442914e-01 -5.68488678e-16
  -8.52277813e-01  8.40541619e-01]]
[0.83071084 0.88813661 1.0445667  1.04645814]
[array([[4, 5],
       [3, 5],
       [2, 5],
       [4, 6],
       [3, 6],
       [2, 6],
       [1, 5],
       [1, 6]]), array([[4, 5],
       [3, 5],
       [2, 5],
       [4, 6],
       [3, 6],
       [2, 6],
       [1, 5],
       [1, 6]]), array([[4, 5],
       [3, 5],
       [2, 5],
       [4, 6],
       [3, 6],
       [2, 6],
       [1, 5

In [20]:
data1[0]

array([0.83071084, 0.88813661, 1.0445667 , 1.04645814])

However, in realistic calculations the number of configurations may be very large, so the calculations will be more expensive for the two reasons:

- the bare number of configurations will increase (so the size of the CI amplitude matrices will increase)

- more critically: since the excitations from "deeper" levels will be included, the size of the Slater determinant will be larger - more electrons will need to be included explicitly.

Reading with CI threshold:

In [21]:
info, mo1, data1 = DFTB_methods.read_dftb_orbital_info({"nstates":5, "orbital_space":None, 
                                                        "source_directory":"calc2",
                                                        "ci_threshold":0.01})
print(data1[1])
print(data1[2])
print(info)

[array([[4, 5]]), array([[3, 5],
       [2, 6],
       [1, 5]]), array([[2, 5],
       [3, 6],
       [1, 6]]), array([[4, 6]])]
[array([-1.]), array([-0.99267763,  0.0660317 , -0.03828367]), array([-0.89191693,  0.41936233, -0.07312469]), array([1.])]
{'nocc': np.int64(4), 'nelec': np.int64(8), 'nao': 6, 'nmo': 6, 'nci': 4, 'nact': 6, 'actual_orbital_space': [1, 2, 3, 4, 5, 6], 'min_occ': np.int64(1), 'max_occ': np.int64(4), 'min_vir': np.int64(5), 'max_vir': np.int64(6)}


## 5. Putting everything together
<a name="5"></a>[Back to TOC](#toc)

### 5.1. Develop the function
<a name="5.1"></a>[Back to TOC](#toc)

In [22]:
class tmp:
    pass

def dftb_compute_adi(q, params, full_id):
    
    Id = Cpp2Py(full_id)
    itraj = Id[-1]

    # Sanity check on params structure
    if not isinstance(params, (list, tuple)):
        raise TypeError(
            "params must be a list (or tuple) of per-trajectory dictionaries; "
            f"got {type(params)}"
        )

    if itraj >= len(params):
        raise IndexError(
            f"Trajectory index itraj={itraj} out of range for params (len={len(params)})"
        )

    if not isinstance(params[itraj], dict):
        raise TypeError(
            f"params[{itraj}] must be a dictionary; got {type(params[itraj])}"
        )

    # MATRIX format
    coords = q.col(itraj)
    
    # This should be the nparray version of coords
    coordinates = data_conv.MATRIX2nparray(coords, float)

    
    dt = params[itraj].get("dt", "41.0") # 1 fs default
    
    dftb_run_params = params[itraj].get("dftb_run_params", {})
    atom_labels = params[itraj].get("atom_labels", None )
    dftb_exe = params[itraj].get("dftb_exe", "dftb+")
    wd = params[itraj].get("working_directory", "wd")
    
    gen_file = dftb_run_params.get("gen_file", "x1.gen")
    sk_prefix = dftb_run_params.get("sk_prefix", "../mio/FinalSK/")
    
    odin_max_ang_mom = params[itraj].get("odin_max_ang_mom", {"H":1, "O":2} )
    odin_exe = params[itraj].get("odin_exe", "odin")
    
    orbital_space = params[itraj]["orbital_space"]
    
    nstates = dftb_run_params.get("NrOfExcitations", 1) + 1
    
    is_first_time = params[itraj].get("is_first_time", True)
    nelec_act_space = params[itraj].get("nelec_act_space", None)
    
    ci_threshold = params[itraj].get("ci_threshold", 0.01)
    

    # print("================ RUN DFTB+ =================\n")
    # Make input for DFTB+ SCF calculations
    # make sure that the path to the Slater-Koster files directory in dftb_params
    # is given considering that the calculations will be conducted in the 
    # `working_directory` created by the function below
    DFTB_methods.make_dftb_input(dftb_params)
    prms1 = { "labels" : atom_labels,
          "exe": dftb_exe,
          "dftb_run_params" : dftb_params,
          "working_directory" : wd,
          "gen_file" : gen_file 
        }
    
    # Create the working directory and run the calculation
    # this will also create an appropriate gen file 
    
    #print(atom_labels)
    #print(coordinates)
    DFTB_methods.run_dftb(coordinates, prms1)

    
    # print("================ RUN ODIN =================\n")
    coordinates_prev = None
    if is_first_time:
        # On the first step, assume the current properties are as the previous
        coordinates_prev = copy.deepcopy(coordinates)
    else:
        # Otherwise, retrieve the previously-stored data
        coordinates_prev = copy.deepcopy(params[itraj]["coordinates_prev"])
        
    double_labels = atom_labels*2
    double_coords = np.concatenate( (coordinates_prev, coordinates), axis=0)

    # Let's reuse the working directory of DFTB+ for ODIN as well
    # Go into that directory
    os.chdir(wd)
    
    # Make double geometry gen file
    DFTB_methods.write_dftb_gen("doubled_geometry.gen", double_labels, double_coords/units.Angst)

    # Make input for Odin:
    odin_params = {
        "ODIN_EXE": odin_exe,
        "filename":"doubled_geometry.gen",
        "slakos_prefix" : sk_prefix,
        "max_ang_mom": odin_max_ang_mom,
    }
    DFTB_methods.create_odin_inp(odin_params)

    # Run Odin
    DFTB_methods.run_odin(odin_params)

    # Go back to the original directory
    os.chdir("../")

    # Read the DFTB+ output
    # print("================ READ DFTB+ =================\n")
    read_params = {"nstates":nstates, "orbital_space":orbital_space, 
                   "source_directory":wd, "ci_threshold":ci_threshold }
    info, MO_curr, data_curr = DFTB_methods.read_dftb_orbital_info(read_params)
    print(F"is_first_time = {is_first_time}")
    print(F"info = {info}")
    print(F"data_curr = {data_curr}")
    print(F"MO_curr = {MO_curr}")

    ## Overlap matrix
    # print("================ READ AO overlaps =================\n")
    ndim = info["nmo"]
    S = DFTB_methods.read_overlap_matrix(F'{wd}/oversqr.dat', 2*ndim)
    
    s_ao_curr = S[ndim:2*ndim, ndim:2*ndim]
    st_ao = S[:ndim, ndim:2*ndim]
    
    # Check MO normalization:
    mat = MO_curr.T @ s_ao_curr @ MO_curr
    print("Checking MO orthonormalizaiton")
    DFTB_methods.check_unity_deviation(mat)
    
    #================= Construct active space ==================
    active_space = None
    if nelec_act_space is None:
        active_space = info["actual_orbital_space"]
    else:
        min_indx = info["nocc"] - nelec_act_space//2 + 1
        if min_indx > info["min_occ"]:
            min_elec = ((info["nocc"] - info["min_occ"]) + 1 )*2
            raise ValueError(f"The `nelec_act_space` should be at least { min_elec }")
        active_space = list(range(min_indx, info["nmo"]+1))

    # Get the properties at the previous time-steps
    MO_prev, data_prev = None, None
    if is_first_time:
        # On the first step, assume the current properties are as the previous
        MO_prev = copy.deepcopy(MO_curr)
        data_prev = copy.deepcopy(data_curr)
    else:
        # Otherwise, retrieve the previously-stored data
        MO_prev = copy.deepcopy(params[itraj]["MO_prev"])
        data_prev = copy.deepcopy(params[itraj]["data_prev"])

    # Do the calculations - time-overlaps, energies, and Hvib
    obj = tmp()
    obj.ham_adi = CMATRIX(nstates, nstates)
    obj.nac_adi = CMATRIX(nstates, nstates)
    obj.hvib_adi = CMATRIX(nstates, nstates)
    obj.basis_transform = CMATRIX(nstates, nstates)
    obj.time_overlap_adi = CMATRIX(nstates, nstates)
    obj.overlap_adi = CMATRIX(nstates, nstates)
    
    #======================= MO ===============================
    # MO overlaps
    st_mo_orb = MO_prev.T @ st_ao @ MO_curr

    # Make it doubled - block-matrix
    st_mo = np.kron(np.eye(2), st_mo_orb)

    #================= Compute CI time-overlaps =============
    ovlp_params = {"homo_indx":info["nocc"],
                   "nocc":info["nocc"] - 1,
                   "nvirt":info["nmo"] - info["nocc"],
                   "nelec":info["nelec"], "nstates":nstates,
                   "active_space":active_space
                   }
    print(ovlp_params)
    st_ci = ci.overlap(st_mo, data_prev, data_curr, ovlp_params)

    #=============== Now, populate the allocated matrices ======================
    for istate in range(nstates):
        energ = 0.0
        if istate > 0:
            energ = float(0.5 * (data_prev[0][istate-1] + data_curr[0][istate-1]))

        obj.ham_adi.set(istate, istate, energ * (1.0 + 0.0j))
        obj.hvib_adi.set(istate, istate, energ * (1.0 + 0.0j))
        obj.basis_transform.set(istate, istate, 1.0 + 0.0j)  # assume identity

        for jstate in range(nstates):
            obj.time_overlap_adi.set(istate, jstate, float(st_ci[istate, jstate]) * (1.0 + 0.0j))

    # Update the Hvib:
    for istate in range(nstates):
        for jstate in range(istate + 1, nstates):
            dij = (obj.time_overlap_adi.get(istate, jstate) - obj.time_overlap_adi.get(jstate, istate)) / (2.0 * dt)
            obj.hvib_adi.set(istate, jstate, dij * (0.0 - 1.0j))
            obj.hvib_adi.set(jstate, istate, dij * (0.0 + 1.0j))

    # Now, make the current the previous and reset the flag `is_first_time` to False
    # Note - we directly modify the input parameters
    params[itraj]["MO_prev"] = copy.deepcopy(MO_curr)
    params[itraj]["data_prev"] = copy.deepcopy(data_curr)
    params[itraj]["coordinates_prev"] = copy.deepcopy(coordinates)
    params[itraj]["is_first_time"] = False

    return obj

### 5.2. Test the funciton
<a name="5.2"></a>[Back to TOC](#toc)

In [23]:
os.chdir("/home/alexvakimov/CCCT/Tutorials_Libra/11_program_specific_methods/4_dftbplus_methods/3_workflow")
labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", 0)

print(labels)
print(q)

wd = "dftb_results"

# Create working directory, if doesn't exist
if not os.path.exists(wd):
    os.mkdir(wd)

dftb_params = {
    "gen_file" : "x1.gen",
    "sk_prefix" : "../mio/FinalSK/",
    "Driver" : "{}",
    "MaxAngularMomentum" : """{ C = "p"
                                H = "s"
                     }
                     """,
    "Symmetry" : "Singlet",
    "NrOfExcitations" : 5,
    "StateOfInterest" : 1,
    "WriteSPTransitions" : "Yes",
    "WriteXplusY" : "Yes",
    "WriteXplusYAscii" : "Yes",
    "StateCouplings" : "{0 3}",
    
    "WriteAutotestTag" : "Yes",
    "WriteHS" : "No",
    "WriteEigenvectors" : "Yes",
    "EigenvectorsAsText" : "Yes",
    "PrintForces" : "Yes",
    "Filling" : """Fermi { Temperature [K] = 0.5 }
                """
}

params_elem = {"atom_labels":labels, "timestep":0, "is_first_time":True,
               "dftb_exe":"/home/alexvakimov/opt/dftb+/bin/dftb+", 
               "dftb_run_params": dftb_params,
               "working_directory":"dftb_workflow",
               
               "odin_exe": "/home/alexvakimov/SOFTWARE/odin/odin",
               "odin_max_ang_mom" : { "C":2, "H":1 },
               "orbital_space" : None,
               
               "dt":1.0*units.fs2au,
               "nelec_act_space":None,
               "ci_threshold":0.01
              }

# For 1 trajectory
params = [ dict(params_elem), ]

print(params)

# Emulates 1 trajectory
full_id = Py2Cpp_int([0, 0])

# Do the first 5 steps 
for i in range(5):
    print(F"======== Iteration {i} ==============")
    labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", i)
    params[0]["timestep"] = i
    
    obj = dftb_compute_adi(q, params, full_id)        
    obj.ham_adi.show_matrix(F"{wd}/ham_adi_{i}.txt")
    obj.hvib_adi.show_matrix(F"{wd}/hvib_adi_{i}.txt")
    obj.time_overlap_adi.real().show_matrix(F"{wd}/st_adi_{i}.txt")

['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
[{'atom_labels': ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H'], 'timestep': 0, 'is_first_time': True, 'dftb_exe': '/home/alexvakimov/opt/dftb+/bin/dftb+', 'dftb_run_params': {'gen_file': 'x1.gen', 'sk_prefix': '../mio/FinalSK/', 'Driver': '{}', 'MaxAngularMomentum': '{ C = "p"\n                                H = "s"\n                     }\n                     ', 'Symmetry': 'Singlet', 'NrOfExcitations': 5, 'StateOfInterest': 1, 'WriteSPTransitions': 'Yes', 'WriteXplusY': 'Yes', 'WriteXplusYAscii': 'Yes', 'StateCouplings': '{0 3}', 'WriteAutotestTag': 'Yes', 'WriteHS': 'No', 'WriteEigenvectors': 'Yes', 'EigenvectorsAsText': 'Yes', 'PrintForces': 'Yes', 'Filling': 'Fermi { Temperature [K] = 0.5 }\n                '}, 'working_directory': 'dftb_workflow', 'odin_exe': '/home

Checking MO orthonormalizaiton
--- Matrix Deviation Analysis (56x56) ---
Max Absolute Deviation: 1.57e-06
{'homo_indx': np.int64(28), 'nocc': np.int64(27), 'nvirt': np.int64(28), 'nelec': np.int64(56), 'nstates': 6, 'active_space': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]}


/tmp/ipykernel_214202/1109037634.py:193: ComplexWarning: Casting complex values to real discards the imaginary part
  obj.time_overlap_adi.set(istate, jstate, float(st_ci[istate, jstate]) * (1.0 + 0.0j))


======== Iteration 1 ==============
DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

-------------------------------------------------

======== Iteration 2 ==============
DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

-------------------------------------------------

======== Iteration 3 ==============
DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

-------------------------------------------------

======== Iteration 4 ==============
DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

-------------------------------------------------

### 5.3. Test Libra implementation
<a name="5.3"></a>[Back to TOC](#toc)
<a name="dftb_compute_adi-1"></a>

In [24]:
os.chdir("/home/alexvakimov/CCCT/Tutorials_Libra/11_program_specific_methods/4_dftbplus_methods/3_workflow")
labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", 0)

print(labels)
print(q)

wd = "dftb_results2"

# Create working directory, if doesn't exist
if not os.path.exists(wd):
    os.mkdir(wd)

dftb_params = {
    "gen_file" : "x1.gen",
    "sk_prefix" : "../mio/FinalSK/",
    "Driver" : "{}",
    "MaxAngularMomentum" : """{ C = "p"
                                H = "s"
                     }
                     """,
    "Symmetry" : "Singlet",
    "NrOfExcitations" : 5,
    "StateOfInterest" : 1,
    "WriteSPTransitions" : "Yes",
    "WriteXplusY" : "Yes",
    "WriteXplusYAscii" : "Yes",
    "StateCouplings" : "{0 3}",
    
    "WriteAutotestTag" : "Yes",
    "WriteHS" : "No",
    "WriteEigenvectors" : "Yes",
    "EigenvectorsAsText" : "Yes",
    "PrintForces" : "Yes",
    "Filling" : """Fermi { Temperature [K] = 0.5 }
                """
}

params_elem = {"atom_labels":labels, "timestep":0, "is_first_time":True,
               "dftb_exe":"/home/alexvakimov/opt/dftb+/bin/dftb+", 
               "dftb_run_params": dftb_params,
               "working_directory":"dftb_workflow2",
               
               "odin_exe": "/home/alexvakimov/SOFTWARE/odin/odin",
               "odin_max_ang_mom" : { "C":2, "H":1 },
               "orbital_space" : None,
               
               "dt":1.0*units.fs2au,
               "nelec_act_space":None,
               "ci_threshold":0.01
              }

# For 1 trajectory
params = [ dict(params_elem), ]

print(params)

# Emulates 1 trajectory
full_id = Py2Cpp_int([0, 0])

# Do the first 5 steps 
for i in range(5):
    print(F"======== Iteration {i} ==============")
    labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", i)
    params[0]["timestep"] = i
    
    obj = DFTB_methods.dftb_compute_adi(q, params, full_id)        
    obj.ham_adi.show_matrix(F"{wd}/ham_adi_{i}.txt")
    obj.hvib_adi.show_matrix(F"{wd}/hvib_adi_{i}.txt")
    obj.time_overlap_adi.real().show_matrix(F"{wd}/st_adi_{i}.txt")

['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
[{'atom_labels': ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H'], 'timestep': 0, 'is_first_time': True, 'dftb_exe': '/home/alexvakimov/opt/dftb+/bin/dftb+', 'dftb_run_params': {'gen_file': 'x1.gen', 'sk_prefix': '../mio/FinalSK/', 'Driver': '{}', 'MaxAngularMomentum': '{ C = "p"\n                                H = "s"\n                     }\n                     ', 'Symmetry': 'Singlet', 'NrOfExcitations': 5, 'StateOfInterest': 1, 'WriteSPTransitions': 'Yes', 'WriteXplusY': 'Yes', 'WriteXplusYAscii': 'Yes', 'StateCouplings': '{0 3}', 'WriteAutotestTag': 'Yes', 'WriteHS': 'No', 'WriteEigenvectors': 'Yes', 'EigenvectorsAsText': 'Yes', 'PrintForces': 'Yes', 'Filling': 'Fermi { Temperature [K] = 0.5 }\n                '}, 'working_directory': 'dftb_workflow2', 'odin_exe': '/hom

/home/alexvakimov/SOFTWARE/libra/_build/src/libra_py/packages/dftbplus/methods.py:2692: ComplexWarning: Casting complex values to real discards the imaginary part
  obj.time_overlap_adi.set( i, j, float(st_ci[i, j])  )


======== Iteration 1 ==============
DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

-------------------------------------------------

        4             129
>> Stratmann converged

Total Energy:                      -22.8151408633 H         -620.8316 eV
Extrapolated to 0K:                -23.2684813429 H         -633.1676 eV
Total Mermin free energy:          -22.8151408633 H         -620.8316 eV
Force related energy:              -23.2684813429 H         -633.1676 eV


--------------------------------------------------------------------------------
DFTB+ running times                          cpu [s]             wall clock [s]
--------------------------------------------------------------------------------
Global initialisation                  +       0.05 (  1.3%)       0.00 (  1.3%)
Pre-SCC initialisation                 +       0.10 (  2.9%)       0.01 (  3.0%)
SCC                                    +       0.88 ( 24.4%)       0.06 ( 25.5%)
Post-SCC processing                    +       2.26 ( 62.5%)       0.14 ( 61.5%)
Post-geometry optimisation             +       0.02 (  0.4%)       0.00 (  0.4%)
---------

======== Iteration 4 ==============
DFTB+ GEN file written to x1.gen
|===============================================================================
|
|  DFTB+ development version (commit: f17f4be1, base: 24.1)
|
|  Copyright (C) 2006 - 2025  DFTB+ developers group
|
|===============================================================================
|
|  When publishing results obtained with DFTB+, please cite the following
|  reference:
|
|    Recent Developments in DFTB+, a Software Package for Efficient Atomistic
|    Quantum Mechanical Simulations, J. Phys. Chem. A 129, 5373−5390 (2025).
|    [https://doi.org/10.1021/acs.jpca.5c01146]
|
|  You should also cite additional publications crediting the parametrization
|  data you use. Please consult the documentation of the SK-files for the
|  references.
|
|===============================================================================

Reading input file 'dftb_in.hsd'
Parser version: 14

-------------------------------------------------